# Coastal flood step 02: direct damages (maximum_scenario)

Runs direct damage calculations for this set scenario.


In [ ]:
import pandas
import subprocess
from pathlib import Path

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")


In [ ]:
output_path = base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario"
intersections_input_path = base_path / "dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
damage_curves_path = data_root / "damage_curves"

results_directory = Path(output_path)
results_directory.mkdir(parents=True, exist_ok=True)
intersections_input_path.mkdir(parents=True, exist_ok=True)


In [ ]:
network_csv = data_root / "networks/network_layers_hazard_intersections_details.csv"
coastal_rasters_input_csv = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
damage_curves_csv = damage_curves_path / "asset_damage_curve_mapping.csv"
hazard_damage_parameters_csv = damage_curves_path / "hazard_damage_parameters.csv"
network_assets_root = data_root / "networks"


In [ ]:
coastal_rasters_table = pandas.read_csv(coastal_rasters_input_csv)
coastal_rasters_table["fname"] = coastal_rasters_table["path"]
coastal_rasters_table["hazard"] = "coastal"
coastal_rasters_output_csv = results_directory / "coastal_flood_rasters_for_intersections.csv"
coastal_rasters_table.to_csv(coastal_rasters_output_csv, index=False)

hazard_csv = coastal_rasters_output_csv

print("Hazard layers file:", coastal_rasters_output_csv)
print("Summary hazard file:", hazard_csv)


In [ ]:
script_path = base_path / "scripts/analysis/damage_calculations_coastal_sensitivity.py"
hazard_layers_name = coastal_rasters_output_csv.stem
damage_results_folder = results_directory / "direct_damages"
damage_results_folder.mkdir(parents=True, exist_ok=True)

sensitivity_csv = results_directory / "sensitivity_parameters.csv"
pandas.DataFrame(
    [{"cost_uncertainty_parameter": 1.0, "damage_uncertainty_parameter": 1.0}]
).to_csv(sensitivity_csv, index=False)
print("Using coastal sensitivity maximum_scenario: cost_uncertainty_parameter=1.0, damage_uncertainty_parameter=1.0")

asset_data_details = pandas.read_csv(network_csv)

for asset_info in asset_data_details.itertuples():
    asset_gpkg_file = network_assets_root / asset_info.path
    if not asset_gpkg_file.exists():
        raise FileNotFoundError(f"Could not find asset file: {asset_gpkg_file}")
    intersection_file = intersections_input_path / f"{asset_info.asset_gpkg}_splits__{hazard_layers_name}__{asset_info.asset_layer}.geoparquet"
    output_file = damage_results_folder / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}" / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    args = [
        "python", str(script_path),
        "--network-csv", str(network_csv),
        "--hazard-csv", str(hazard_csv),
        "--sensitivity-csv", str(sensitivity_csv),
        "--sensitivity-id", "0",
        "--asset-gpkg-file", str(asset_gpkg_file),
        "--asset-gpkg-label", str(asset_info.asset_gpkg),
        "--asset-layer", str(asset_info.asset_layer),
        "--damage-curve-mapping-csv", str(damage_curves_csv),
        "--damage-threshold-uplift-csv", str(hazard_damage_parameters_csv),
        "--damage-curves-dir", str(damage_curves_path),
        "--intersection", str(intersection_file),
        "--output-path", str(output_file),
    ]
    print(args)
    run_result = subprocess.run(args, capture_output=True, text=True)
    if run_result.returncode != 0:
        print(run_result.stdout)
        print(run_result.stderr)
        run_result.check_returncode()

print("Finished direct damage calculations")
